<details><summary style="display:list-item; font-size:16px; color:blue;">Jupyter Help</summary>
    
Having trouble testing your work? Double-check that you have followed the steps below to write, run, save, and test your code!

[Click here for a walkthrough GIF of the steps below](https://static-assets.codecademy.com/Courses/ds-python/jupyter-help.gif)

Run all initial cells to import libraries and datasets. Then follow these steps for each question:

1. Add your solution to the cell with `## YOUR SOLUTION HERE ##`.
2. Run the cell by selecting the `Run` button or the `Shift`+`Enter` keys.
3. Save your work by selecting the `Save` button, the `command`+`s` keys (Mac), or `control`+`s` keys (Windows).
4. Select the `Test Work` button at the bottom left to test your work.

![Screenshot of the buttons at the top of a Jupyter Notebook. The Run and Save buttons are highlighted](https://static-assets.codecademy.com/Paths/ds-python/jupyter-buttons.png)

#### Setup Cell

Run the setup cell below to import required libraries and initialize the language model.

`MessagesState` is a built-in conversation state with an append-only `messages` list. `ToolNode` is a pre-built node that automatically executes tool calls, and the `@tool` decorator converts a Python function into a callable tool. `HumanMessage` represents user input in the conversation.

The language model `llm` is initialized using `gpt-4o` with a temperature of `0.5`.

**Note:** The OpenAI API key is already configured in this workspace, so no additional setup is needed. When working in an external environment, refer to the [OpenAI API](https://platform.openai.com/docs/quickstart?language=python) documentation for details on configuring your API key.

In [ ]:
# Third-party imports
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from typing import Literal
import warnings

warnings.filterwarnings("ignore")

# Initialize the language model
llm = ChatOpenAI(model="gpt-4o", temperature=0.5)

#### Checkpoint 1/5: Define the Tools and Draft Store

Let's set up the utility tools and storage that our publishing assistant will use during conversations.

- Create an empty dictionary called `drafts_store` to hold saved drafts. Then define two tools using the `@tool` decorator.

- Define a function `count_words` that takes a `text` parameter, splits it into words, and returns a formatted string showing the word count.

- Define a function `save_draft` that takes `title` and `content` parameters, saves `content` to `drafts_store` under the `title` key, and returns a formatted string showing the total number of drafts.

**Note:** Don't forget to run the cell and save the notebook before selecting `Test Work`. Open the `Jupyter Help` toggle at the top of the notebook for more details.

In [ ]:
# Create a dictionary to store saved drafts
drafts_store = {}

# Define a tool that counts words in a given text
@tool
def count_words(text: str) -> str:
    """Count the number of words in the given text."""
    return f"Word count: {len(text.split())}"

# Define a tool that saves a draft with a title
@tool
def save_draft(title: str, content: str) -> str:
    """Save a content draft with a title for later reference."""
    drafts_store[title] = content
    return f"Draft '{title}' saved. Total drafts: {len(drafts_store)}"

#### Checkpoint 2/5: Bind the Tools to the LLM

With our tools defined, let's register them with the language model so the assistant can call them during conversations.

- Create a list called `tools` containing the `count_words` and `save_draft` functions.

- Call `bind_tools()` on `llm`, passing in the `tools` list, and assign the result to `llm_with_tools`.

**Note:** Don't forget to run the cell and save the notebook before selecting `Test Work`. Open the `Jupyter Help` toggle at the top of the notebook for more details.

In [ ]:
# Create a list of tools for the assistant
tools = [count_words, save_draft]

# Bind the tools to the language model
llm_with_tools = llm.bind_tools(tools)

#### Checkpoint 3/5: Define the Assistant Node and Routing Function
 
Now, let's create the assistant node and a routing function that accepts a `state` parameter of type `MessagesState` to process messages and decide whether to execute tools or end the turn.

- **Assistant:** Define a function `assistant_node` that invokes `llm_with_tools` with the state's `messages`, and returns a dictionary with a `messages` key containing the response as a list.

- **Routing:** Define a function `should_use_tools` that returns either `"tools"` or `"end"` using a `Literal` type.

   - Retrieve the latest message from the state's `messages` list and store it in `last_message`.

   - If `last_message` has a `tool_calls` attribute and it is non-empty, return `"tools"`; otherwise, return `"end"`.

**Note:** Don't forget to run the cell and save the notebook before selecting `Test Work`. Open the `Jupyter Help` toggle at the top of the notebook for more details.

In [ ]:
# Define the assistant node that processes messages
def assistant_node(state: MessagesState):
    """Publishing assistant that responds and uses tools."""
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

# Define the routing function to check for tool calls
def should_use_tools(state: MessagesState) -> Literal["tools", "end"]:
    """Route to tools if the assistant made tool calls."""
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return "end"

#### Checkpoint 4/5: Build the Graph with a Tool-Calling Loop

With the assistant node and routing function ready, let's construct the graph that connects the assistant to its tools.

- Create a `StateGraph` instance with `MessagesState` as its state type and assign it to `builder`. 

- Add two nodes: `assistant` (using `assistant_node`) and `tools` (using a `ToolNode` initialized with the `tools` list). 

- Wire the execution flow with an edge from `START` to `assistant`, a conditional edge from `assistant` using `should_use_tools` (mapping `"tools"` to `tools` and `"end"` to `END`), and an edge from `tools` back to `assistant` to create the loop.

**Note:** Don't forget to run the cell and save the notebook before selecting `Test Work`. Open the `Jupyter Help` toggle at the top of the notebook for more details.

In [ ]:
# Initialize the graph with the defined state schema
builder = StateGraph(MessagesState)

# Add nodes to the graph
builder.add_node("assistant", assistant_node)
builder.add_node("tools", ToolNode(tools))

# Connect the nodes using edges
builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", should_use_tools, {"tools": "tools", "end": END})
builder.add_edge("tools", "assistant")

#### Checkpoint 5/5: Compile the Graph with Memory

Let's add persistent memory so the assistant can maintain conversation history, then compile the graph into a runnable pipeline.

- Create a `MemorySaver` instance and assign it to `memory`.

- Call `compile()` on `builder` with `checkpointer` set to `memory`, and assign the result to `publishing_assistant`.

**Note:** Don't forget to run the cell and save the notebook before selecting `Test Work`. Open the `Jupyter Help` toggle at the top of the notebook for more details.

In [ ]:
# Initialize an in-memory checkpointer to save graph state
memory = MemorySaver()

# Compile the graph with the checkpointer
publishing_assistant = builder.compile(checkpointer=memory)

#### Run a Multi-Turn Conversation

With the assistant compiled, let's test it in a multi-turn conversation that exercises content generation, tool use, and memory recall.

A `config` dictionary with a `thread_id` is already provided to maintain conversation history across turns.

The conversation proceeds through four turns:
- **Turn 1** asks the assistant to generate a short blog post on a given topic.
- **Turn 2** requests a word count of the generated post, triggering the `count_words` tool.
- **Turn 3** asks the assistant to save the post using the `save_draft` tool.
- **Turn 4** tests memory by asking the assistant to recall the user's name and topic from earlier turns.

In [ ]:
config = {"configurable": {"thread_id": "editor-session-1"}}

# Turn 1: Generate content
print("=== Turn 1: Generate Content ===")
r1 = publishing_assistant.invoke(
    {"messages": [HumanMessage(content=
        "Hi, I'm Alex from the content team. "
        "Write a short blog post (<300 words) about"
        "ethical concerns around AI-generated misinformation.")]},
    config=config
)
print(f"Assistant: {r1['messages'][-1].content[:300]}...")

# Turn 2: Word count
print("\n=== Turn 2: Check Word Count ===")
r2 = publishing_assistant.invoke(
    {"messages": [HumanMessage(content="How many words was that blog post?")]},
    config=config
)
print(f"Assistant: {r2['messages'][-1].content}")

# Turn 3: Save draft
print("\n=== Turn 3: Save Draft ===")
r3 = publishing_assistant.invoke(
    {"messages": [HumanMessage(content="Save that blog post as 'AI Ethical Concerns'.")]},
    config=config
)
print(f"Assistant: {r3['messages'][-1].content}")

# Turn 4: Memory test
print("\n=== Turn 4: Memory Check ===")
r4 = publishing_assistant.invoke(
    {"messages": [HumanMessage(content="What is my name and what topic did I ask about?")]},
    config=config
)
print(f"Assistant: {r4['messages'][-1].content}")

<details> 
    
<summary><strong>Summary</strong></summary> 

In this exercise, we combined all the LangGraph concepts from this module: state management, conditional routing, tool calling, and persistent memory. This pattern forms the foundation for production-ready AI assistants.

</details>